To align with a company like Cetaris, build a Fleet Maintenance Management System (FMMS).

Cetaris focuses on Asset Lifecycle Management. Mostly care about the relationship between parts, labor costs, downtime, and PM (Preventive Maintenance) schedules.

Here is the step-by-step roadmap to building this from scratch, designed to look like a professional industry solution.

Step 1: The "Physical Logic" Data Generation
Instead of one flat file, we will create a Relational Structure. Real maintenance data is split across different tables. We will simulate three core tables.

The Plan:

- Assets Table: Static info (Vehicle ID, Year, Make, Model).
- Telemetry Table: Real-time sensor "pings" (Odometer, Fuel Level, Engine Hours).
- Work Orders Table: Historical repairs (Part Replaced, Labor Hours, Cost, Failure Type).



In [20]:
import pandas as pd
import numpy as np

# 1. Assets: 1000 Vehicles
n_vehicles = 1000
df_assets = pd.DataFrame({
    'asset_id': [f'TRUCK_{i:03d}' for i in range(n_vehicles)],
    'type': np.random.choice(['Heavy Duty', 'Medium Duty', 'Light Duty'], n_vehicles),
    'purchase_year': np.random.randint(2015, 2023, n_vehicles)
})

# 2. Telemetry: Daily snapshots for 1 year (365,000 rows)
# This is where the ML "signals" live. 
# Logic: Odometer should increase by ~100-500 miles/day.

In [21]:
print(df_assets.head())
print(df_assets.shape)

    asset_id        type  purchase_year
0  TRUCK_000  Light Duty           2019
1  TRUCK_001  Heavy Duty           2020
2  TRUCK_002  Heavy Duty           2015
3  TRUCK_003  Light Duty           2016
4  TRUCK_004  Heavy Duty           2015
(1000, 3)


**Step 2: Injecting "The Failure Pattern"**
This is the "Secret Sauce." We must hide a failure pattern in the data that isn't obvious.

Logic for your Simulation:

The "Heat" Pattern: If avg_ambient_temp is > 90°F AND engine_load is > 80% for 3 consecutive days, the probability of a "Cooling System" failure in the next 7 days increases by 40%.

The "Vibration" Pattern: Create a vibration_index. If the standard deviation of vibration increases over 5 days, a "Suspension" failure is imminent.

**Step 3: Feature Engineering (The Cetaris Way)**
In a resume interview, they will ask: "How did you handle the time element?"
You must move from raw sensors to Calculated Metrics:

Utilization Rate: (Miles driven / Days since last service).

Downtime Impact: Total hours the asset was "Status: Down" in the last 90 days.

Mean Time Between Failures (MTBF): The average operating time between work orders.

**Step 4: The Modeling Strategy**
Instead of just "Predicting Failure," build a Multi-Class Problem:

Class 0: Healthy.

Class 1: Needs Inspection (Minor issue).

Class 2: Immediate Critical Failure (Stop the truck!).

**Step 5: Explainability (SHAP)**
Because you are directing "Maintenance," you need to tell the mechanic what to look at. Use SHAP to generate a "Reason Code" for every prediction.

Example Output: "Vehicle TRUCK_042 has a 88% failure risk. Primary Driver: High Oil Temperature (45% contribution)."

In [22]:
import pandas as pd
import numpy as np

# Settings
n_assets = 100
days = 300
data_list = []
work_orders = []

for asset_id in range(n_assets):
    # Static Asset Info
    a_id = f'ASSET_{asset_id:03d}'
    
    # Starting values
    odometer = np.random.randint(10000, 50000)
    
    for day in range(days):
        # 1. Base Telemetry
        daily_miles = np.random.randint(100, 400)
        odometer += daily_miles
        ambient_temp = np.random.normal(75, 15) # Some days are hot
        engine_load = np.random.uniform(50, 95)
        vibration_index = np.random.normal(10, 1) # Normal vibration is steady
        
        # 2. Injecting the "Heat Pattern" Logic
        # We track the last 3 days in this loop (conceptually)
        # For simplicity in generation, we trigger a failure if certain conditions met:
        if day > 10:
            # Check for 'Heat' Failure (Heat + Load over time)
            if ambient_temp > 92 and engine_load > 85:
                if np.random.rand() < 0.1: # 10% chance this heat spike causes a WO
                    work_orders.append({'asset_id': a_id, 'date': day + 2, 'type': 'Cooling System'})
            
            # Check for 'Vibration' Failure (Increasing instability)
            # We simulate a "Loose Bolt" or "Suspension" wear
            if day > 250: # Older assets start vibrating more
                vibration_index += np.random.normal(5, 3) 
                if vibration_index > 18:
                    work_orders.append({'asset_id': a_id, 'date': day + 1, 'type': 'Suspension'})

        data_list.append({
            'date': day,
            'asset_id': a_id,
            'odometer': odometer,
            'ambient_temp': ambient_temp,
            'engine_load': engine_load,
            'vibration_index': vibration_index
        })

df_telemetry = pd.DataFrame(data_list)
df_work_orders = pd.DataFrame(work_orders).drop_duplicates(subset=['asset_id', 'date'])

print(f"Telemetry Rows: {len(df_telemetry)}")
print(f"Work Orders (Failures) Generated: {len(df_work_orders)}")

Telemetry Rows: 30000
Work Orders (Failures) Generated: 927


In [ ]:
# #import pandas as pd
# #import numpy as np

# np.random.seed(42)
# days = 300
# n_assets = 100

# data_list = []

# for asset in range(n_assets):
#     odometer = np.random.randint(10000, 50000)
#     for day in range(days):
#         daily_miles = np.random.randint(100, 400)
#         odometer += daily_miles
#         # Simulate a sensor drifting over time
#         oil_pressure = 50 - (day * 0.02) + np.random.normal(0, 2) 
        
#         data_list.append({
#             'date': day,
#             'asset_id': f'ASSET_{asset}',
#             'odometer': odometer,
#             'oil_pressure': oil_pressure,
#             'fuel_consumption': np.random.uniform(5, 12)
#         })

# df_telemetry = pd.DataFrame(data_list)
# print(df_telemetry.head())
# print(df_telemetry.shape)

Work Orders Table: Historical repairs (Part Replaced, Labor Hours, Cost, Failure Type).

In [ ]:
# # Create Work Orders (The Target Variable)
# work_orders = []

# for asset_id in df_telemetry['asset_id'].unique():
#     asset_data = df_telemetry[df_telemetry['asset_id'] == asset_id]
    
#     # Logic: If oil pressure drops below 35 and odometer is high, trigger a failure
#     failure_mask = (asset_data['oil_pressure'] < 38) & (asset_data['odometer'] > 30000)
    
#     # Pick a few random failures to make it realistic
#     failure_days = asset_data[failure_mask]['date'].values
    
#     if len(failure_days) > 0:
#         actual_failure_day = failure_days[0] # Take the first point of failure
#         work_orders.append({
#             'asset_id': asset_id,
#             'failure_date': actual_failure_day,
#             'failure_type': 'Engine System',
#             'repair_cost': np.random.uniform(2000, 5000)
#         })

# df_work_orders = pd.DataFrame(work_orders)


# print(f"Generated {len(df_work_orders)} failure events across {n_assets} assets.")

In [23]:
# Save generated data to the data/raw folder
import os 

os.makedirs('../data/raw', exist_ok=True)
df_assets.to_csv('../data/raw/assets.csv', index=False)
df_telemetry.to_csv('../data/raw/telemetry.csv', index=False)
df_work_orders.to_csv('../data/raw/work_orders.csv', index=False)
